# Lab 3: Dipole Fields and the Geomagnetic Reference Field

| | |
|---|---|
| **Module** | M3 — Magnetic Potential, Magnetization, and the Geomagnetic Field |
| **Estimated time** | ~2.5 hours |
| **Prerequisites** | Lectures M3-1, M3-2; Homework 3, Problems 3.1, 3.4 |
| **Textbook** | Blakely Ch. 4, 5, 8; lecture notes M3 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Implement** the magnetic field components of a point dipole in spherical coordinates
2. **Derive** inclination as a function of latitude from the centered-dipole formula and validate numerically
3. **Compute** the magnetic field elements (F, I, D, X, Y, Z) and convert between representations
4. **Apply** the total-field anomaly approximation and explain how anomaly shape depends on inclination

---

## How to use this notebook

- **`[PROVIDED]`** — run as-is, do not modify
- **`[IMPLEMENT]`** — replace `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check; prints ✓ PASS or ✗ FAIL
- **`[EXPLORE]`** — starting point; modify freely to answer the question above

**Your response:** cells are written-answer questions. Write directly in the notebook.

Hints: `print(hints['key'])` — keys listed at each Part header.


## Background

The magnetic scalar potential of a magnetic dipole with moment $\mathbf{m}$ located at
the origin is

$$
V(\mathbf{r}) = \frac{\mu_0}{4\pi}\frac{\mathbf{m}\cdot\hat{\mathbf{r}}}{r^2}
$$

For an axial dipole $\mathbf{m} = m\,\hat{z}$ this reduces to
$V = \frac{\mu_0}{4\pi}\frac{m\cos\theta}{r^2}$, where $\theta$ is colatitude
(Blakely eq. 5.13). The field components
$\mathbf{B} = -\nabla V$ are

$$
B_r = \frac{\mu_0}{4\pi}\frac{2m\cos\theta}{r^3}, \qquad
B_\theta = \frac{\mu_0}{4\pi}\frac{m\sin\theta}{r^3}, \qquad B_\phi = 0.
$$

Earth's main field is well approximated — to zeroth order — by a **geocentric axial dipole**
(GAD) with moment $M \approx 8.0 \times 10^{22}$ A m$^2$, but with the moment pointing
toward geographic **south** (field lines enter near the geographic north pole).
This reverses the sign of both components compared to the formula above.

The standard geomagnetic field elements at a point are:

| Symbol | Name | Definition |
|--------|------|------------|
| $F$ | Total intensity | $|\mathbf{B}|$ |
| $I$ | Inclination | $\arctan(Z/H)$, positive downward |
| $D$ | Declination | angle of $H$ east of north |
| $X$ | North component | $H\cos D$ |
| $Y$ | East component | $H\sin D$ |
| $Z$ | Vertical (down) | $F\sin I$ |

where $H = \sqrt{X^2+Y^2}$ is the horizontal intensity.

The **total-field anomaly** $\Delta F$ is the difference between the measured field
magnitude and a reference (e.g., IGRF). When the anomaly is small compared to the
regional field $F_0$, to first order:

$$
\Delta F \approx \hat{\mathbf{T}} \cdot \Delta\mathbf{B}
$$

where $\hat{\mathbf{T}} = \mathbf{F}_0/|\mathbf{F}_0|$ is the unit vector in the
direction of the regional field (Blakely eq. 8.1). This approximation is fundamental
to magnetic interpretation.


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

mu0    = 4 * np.pi * 1e-7   # permeability of free space, T·m/A
M_earth = 8.0e22             # Earth's dipole moment magnitude, A·m²
R_earth = 6.371e6            # Earth's mean radius, m

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})
print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
hints = {
    'p1_components': (
        "The field is B = -grad(V). In spherical coordinates with V = (mu0/4pi)*m*cos(theta)/r^2:\n"
        "  B_r = -dV/dr = (mu0/4pi) * 2*m*cos(theta) / r^3\n"
        "  B_theta = -(1/r)*dV/dtheta = (mu0/4pi) * m*sin(theta) / r^3\n"
        "Note: theta is colatitude (0 at north pole, pi at south pole)."
    ),
    'p1_field_lines': (
        "Field lines of the dipole satisfy dr/B_r = r*dtheta/B_theta, which gives\n"
        "r = C * sin^2(theta) for constant C. To plot them, choose several values\n"
        "of C and parametrize theta from 0 to pi."
    ),
    'p2_inclination': (
        "For the GAD, Z (downward) = -B_r (since B_r is outward) and H = |B_theta|.\n"
        "Substituting the dipole formulas gives tan(I) = 2*sin(lat)/cos(lat) = 2*tan(lat).\n"
        "This is the 'geocentric dipole inclination formula' — derive it from your Part 1 functions."
    ),
    'p2_elements': (
        "Standard conversions: H = F*cos(I), Z = F*sin(I), X = H*cos(D), Y = H*sin(D).\n"
        "Inverting: F = sqrt(X^2+Y^2+Z^2), I = arctan(Z/H), D = arctan2(Y, X)."
    ),
    'p3_total_field': (
        "The total-field anomaly is the scalar projection of the anomaly vector onto\n"
        "the regional field direction: delta_F = T_hat . delta_B.\n"
        "In 2D (x=north, z=up) with inclination I and D=0:\n"
        "  T_hat = (cos(I), -sin(I))  [northward, downward => z-component is -sin(I)]\n"
        "  delta_F = delta_B_x * cos(I) + delta_B_z * (-sin(I))\n"
        "Wait -- be careful: Z is downward but z is upward, so B_z(up) = -Z."
    ),
    'p3_dipole_anom': (
        "For a magnetic dipole at the origin, the field at position r_vec is:\n"
        "  B = (mu0/4pi) * [3*(m_hat . r_hat)*r_hat - m_hat] * m / r^3\n"
        "In component form: B_i = (mu0/4pi) * m / r^5 * [3*m_j*r_j*r_i/r^2 ... ]\n"
        "For a 2D geometry (x along profile, z upward, source at (0, -depth)):\n"
        "  r_vec = (x, depth)  [from source to station]\n"
        "  r = sqrt(x^2 + depth^2)"
    ),
}
# print(hints['p1_components'])

---
## Part 1: Magnetic Dipole Field *(Guided — ~40 min)*

We implement the exact dipole field in spherical coordinates, visualize its
structure, and use it as the building block for the rest of the lab.

**Available hints:** `p1_components`, `p1_field_lines`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: implement the two non-zero field components of an axial magnetic dipole.

def dipole_Br(r, theta, m):
    """
    Radial component of a magnetic dipole field.

    Parameters
    ----------
    r : float or ndarray
        Distance from dipole, meters.
    theta : float or ndarray
        Colatitude, radians (0 at north pole, pi at south pole).
    m : float
        Dipole moment magnitude, A·m².  Positive m points along +z.

    Returns
    -------
    B_r : float or ndarray
        Radial field component, Tesla.  Positive = outward.

    Notes
    -----
    B_r = (mu0 / 4*pi) * 2*m*cos(theta) / r^3
    (Blakely eq. 5.15)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def dipole_Btheta(r, theta, m):
    """
    Co-latitude component of a magnetic dipole field.

    Parameters
    ----------
    r : float or ndarray
    theta : float or ndarray
        Colatitude, radians.
    m : float
        Dipole moment, A·m².

    Returns
    -------
    B_theta : float or ndarray
        Colatitude field component, Tesla.  Positive = in direction of increasing theta (southward).

    Notes
    -----
    B_theta = (mu0 / 4*pi) * m*sin(theta) / r^3
    (Blakely eq. 5.15)
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

def _check(label, got, expected, rtol=1e-5, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  ✓ PASS  {label}')
    else:
        print(f'  ✗ FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

_r = 1.0      # unit distance
_m = 1.0      # unit moment
_prefac = mu0 / (4 * np.pi)

# Test 1: B_r at pole (theta=0): should be 2*mu0*m/(4pi*r^3)
_check('B_r at north pole (theta=0)', dipole_Br(_r, 0.0, _m), 2*_prefac)

# Test 2: B_r at equator (theta=pi/2): should be 0
_check('B_r at equator (theta=pi/2)', dipole_Br(_r, np.pi/2, _m), 0.0, atol=1e-20)

# Test 3: B_theta at equator: should be mu0*m/(4pi*r^3)
_check('B_theta at equator', dipole_Btheta(_r, np.pi/2, _m), _prefac)

# Test 4: B_theta at pole: should be 0
_check('B_theta at pole (theta=0)', dipole_Btheta(_r, 0.0, _m), 0.0, atol=1e-20)

# Test 5: at pole, |B_r| = 2 * |B_theta at equator|  (B_pole = 2 * B_equator)
_Br_pole     = dipole_Br(_r, 0.0, _m)
_Bth_equator = dipole_Btheta(_r, np.pi/2, _m)
if np.isclose(_Br_pole, 2*_Bth_equator):
    print('  ✓ PASS  B_pole = 2 * B_equator')
else:
    print('  ✗ FAIL  should have B_pole = 2 * B_equator')

# Test 6: falloff as 1/r^3
_Br_2r = dipole_Br(2*_r, 0.0, _m)
_Br_1r = dipole_Br(_r,   0.0, _m)
_check('1/r^3 falloff', _Br_2r, _Br_1r / 8, rtol=1e-10)

### Question 1.1 — Field line topology

Where (at what colatitude) are $B_r$ and $B_\theta$ equal in magnitude?
What is the inclination of the field at that point?
Verify your answer algebraically from the dipole formulas.

**Your response:**

> *(Write 3–5 sentences. Include the algebraic derivation.)*


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Plot dipole field lines in the meridional (r, theta) plane.
#
# Field lines satisfy r = C * sin^2(theta).
# Choose 8–10 values of C from ~0.3 to ~3 (in units of R_earth) to get
# a good spread from near the surface out to ~3 R_earth.
#
# Also overlay the direction of B at a grid of points using quiver or streamplot.
#
# Requirements:
#   - Axes in units of R_earth
#   - Mark the Earth's surface (circle of radius 1)
#   - Shade the Earth's interior
#   - Show both northern and southern hemispheres

theta = np.linspace(0.01, np.pi - 0.01, 500)
# YOUR CODE HERE

---
## Part 2: The Geocentric Axial Dipole Model *(Supported — ~50 min)*

Earth's field, to first approximation, is a **geocentric axial dipole** (GAD):
a pure dipole centered at Earth's center with moment $M = 8.0\times10^{22}$ A m$^2$
pointing toward geographic **south** (i.e., field lines enter near the north geographic
pole).  The GAD explains roughly 90% of the observed field power.

You will implement the conversion between field components and field elements
(F, I, D, X, Y, Z) and explore the GAD inclination formula.

**Available hints:** `p2_inclination`, `p2_elements`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Implement the GAD field at a given geographic latitude.
# No step-by-step skeleton. Read docstrings and use hints if needed.

def gad_field(lat_deg, r=None):
    """
    Geocentric Axial Dipole field elements at geographic latitude.

    Parameters
    ----------
    lat_deg : float or ndarray
        Geographic latitude, degrees (positive north).
    r : float, optional
        Geocentric radius, meters.  Default: R_earth (surface).

    Returns
    -------
    F : float or ndarray
        Total field intensity, nT.
    I_deg : float or ndarray
        Inclination, degrees (positive = field points downward).
    X : float or ndarray
        North component, nT.
    Z : float or ndarray
        Vertical (downward) component, nT.

    Notes
    -----
    The GAD has M pointing toward geographic south, so the dipole moment
    is m = -M_earth (negative sign flips the direction).
    Colatitude theta = pi/2 - lat (in radians).

    Z (downward) = -B_r (since B_r is radially outward).
    X (northward) = -B_theta (since B_theta is in the +theta = southward direction).
    Y (eastward)  = 0 for the axial dipole.
    F = sqrt(X^2 + Z^2),  I = arctan2(Z, H) where H = |X|.
    Convert F to nT: 1 T = 1e9 nT.
    """
    if r is None:
        r = R_earth
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# At the geographic north pole: I should be +90° (field straight down), F ~ 60,000 nT
_F_np, _I_np, _X_np, _Z_np = gad_field(90.0)
_check('Inclination at north pole = 90°', _I_np, 90.0, atol=0.01)
_check('X (north) at pole = 0', _X_np, 0.0, atol=0.1)
if 50000 < _F_np < 70000:
    print(f'  ✓ PASS  F at north pole = {_F_np:.0f} nT (reasonable range 50000–70000)')
else:
    print(f'  ✗ FAIL  F at north pole = {_F_np:.0f} nT (expected 50000–70000)')

# At the magnetic equator: I should be 0°
_F_eq, _I_eq, _X_eq, _Z_eq = gad_field(0.0)
_check('Inclination at equator = 0°', _I_eq, 0.0, atol=0.01)
_check('Z (vertical) at equator = 0', _Z_eq, 0.0, atol=0.1)

# At south pole: I = -90°
_F_sp, _I_sp, _X_sp, _Z_sp = gad_field(-90.0)
_check('Inclination at south pole = -90°', _I_sp, -90.0, atol=0.01)

# F at poles should be twice F at equator (dipole property)
_check('F_pole = 2 * F_equator', _F_np, 2*_F_eq, rtol=1e-4)

### Question 2.1 — The inclination formula

Starting from your `dipole_Br` and `dipole_Btheta` functions, **derive analytically**
the formula $\tan I = 2\tan\lambda$ (where $\lambda$ is geographic latitude).
Show each step.

Then: at what latitude does $I = 45°$?  Verify numerically with your `gad_field`
function.

**Your response:**

> *(Include the derivation and the numerical check.)*


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Plot GAD inclination (degrees) and total field F (nT) vs. latitude.
# Also plot the *actual* inclination from IGRF at 0° longitude (data below)
# on the same axes and discuss the difference.
#
# Reference IGRF-14 inclination at 0°E, surface, 2025.0 (from NOAA online calculator):
igrf_lat = np.array([-80, -60, -45, -30, -15, 0, 15, 30, 45, 60, 80])
igrf_inc = np.array([-72.4, -66.3, -57.2, -42.8, -23.1, -12.1, 22.3, 46.7, 62.3, 71.4, 79.8])

lat_arr = np.linspace(-90, 90, 360)
# YOUR CODE HERE

### Question 2.2 — GAD vs. IGRF

Where does the GAD inclination agree well with the observed IGRF values,
and where does it deviate most?  What physical features of the geomagnetic
field cause the largest deviations?

**Your response:**

> *(Write 4–5 sentences.)*


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Convert between the (F, I, D) representation and (X, Y, Z) Cartesian.

def field_elements_to_xyz(F, I_deg, D_deg):
    """
    Convert (F, I, D) geomagnetic field elements to (X, Y, Z) components.

    Parameters
    ----------
    F : float or ndarray
        Total field intensity, any consistent units (e.g., nT).
    I_deg : float or ndarray
        Inclination, degrees.  Positive = field points downward.
    D_deg : float or ndarray
        Declination, degrees.  Positive = field points east of north.

    Returns
    -------
    X : northward component, same units as F
    Y : eastward component
    Z : downward component (positive = pointing into Earth)

    Notes
    -----
    H = F*cos(I),  X = H*cos(D),  Y = H*sin(D),  Z = F*sin(I)
    """
    raise NotImplementedError


def xyz_to_field_elements(X, Y, Z):
    """
    Convert (X, Y, Z) geomagnetic components to (F, I, D) elements.

    Parameters
    ----------
    X : float or ndarray
        Northward component, nT (or consistent units).
    Y : float or ndarray
        Eastward component.
    Z : float or ndarray
        Downward component (positive into Earth).

    Returns
    -------
    F : total intensity
    I_deg : inclination, degrees
    D_deg : declination, degrees

    Notes
    -----
    H = sqrt(X^2 + Y^2),  F = sqrt(H^2 + Z^2),
    I = arctan2(Z, H),  D = arctan2(Y, X).
    Use np.degrees to convert.
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# Round-trip test: (F,I,D) -> (X,Y,Z) -> (F,I,D) should recover original
_F0, _I0, _D0 = 55000.0, 63.5, -12.3
_X, _Y, _Z = field_elements_to_xyz(_F0, _I0, _D0)
_Fr, _Ir, _Dr = xyz_to_field_elements(_X, _Y, _Z)
_check('Round-trip F', _Fr, _F0, rtol=1e-6)
_check('Round-trip I', _Ir, _I0, atol=1e-4)
_check('Round-trip D', _Dr, _D0, atol=1e-4)

# At I=90° (north pole), D=0: X=0, Y=0, Z=F
_X2, _Y2, _Z2 = field_elements_to_xyz(60000.0, 90.0, 0.0)
_check('X=0 at I=90°', _X2, 0.0, atol=1e-6)
_check('Z=F at I=90°', _Z2, 60000.0, rtol=1e-6)

# At I=0° (equator), D=0: X=F, Y=0, Z=0
_X3, _Y3, _Z3 = field_elements_to_xyz(30000.0, 0.0, 0.0)
_check('X=F at I=0°', _X3, 30000.0, rtol=1e-6)
_check('Z=0 at I=0°', _Z3, 0.0, atol=1e-6)

---
## Part 3: The Total-Field Anomaly *(Open — ~40 min)*

A survey flown at mid-latitudes detects a magnetic anomaly from a
buried magnetized body.  The measured quantity is the **total-field anomaly**
$\Delta F$, not the individual components of $\Delta\mathbf{B}$.

The goal of this Part is to understand how the *shape* of $\Delta F$ depends
on the inclination $I$ of the regional field — a fundamental limitation of
total-field magnetics and the motivation for reduction-to-pole (covered in Lab 6).

**Available hints:** `p3_total_field`, `p3_dipole_anom`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Geometry and helper function for Part 3.
#
# We work in 2D: x = north, z = upward.
# A uniformly magnetized sphere (equivalent magnetic dipole) is buried
# at depth d below a north-south profile.  Its magnetization is INDUCED:
# it points in the direction of the regional field T_hat.

def dipole_anomaly_2d(x_profile, depth, m_moment, I_deg, D_deg=0.0):
    """
    Vector magnetic anomaly (Bx, Bz) from a buried point dipole in 2D.

    Coordinates: x = north, z = upward.  Dipole at origin, depth d below surface.
    Stations at (x_profile, 0).  Magnetization is along the regional field direction.

    Parameters
    ----------
    x_profile : ndarray
        Station positions along profile, meters.
    depth : float
        Depth to dipole center, meters (positive down).
    m_moment : float
        Dipole moment magnitude, A·m².
    I_deg : float
        Regional field inclination, degrees.
    D_deg : float
        Regional field declination (assumed 0 for 2D).  Not used.

    Returns
    -------
    dBx : ndarray
        North component of anomaly, nT.
    dBz_up : ndarray
        Upward vertical component of anomaly, nT.
    """
    I = np.radians(I_deg)
    # Magnetization direction: (cos I northward, -sin I in z-up) = (cos I, -sin I)
    mx = np.cos(I)    # northward component of unit magnetization
    mz = -np.sin(I)   # upward component (negative because Z-down = -z_up)

    # Vector from dipole to station: r_vec = (x, depth) in (x, z_up) coords
    rx = x_profile            # northward
    rz = depth                # upward distance = +depth because dipole is below
    r2 = rx**2 + rz**2
    r  = np.sqrt(r2)

    # Dipole field:  B_i = (mu0/4pi) * m / r^5 * (3*(m_hat.r)*r_i - m_i*r^2)
    mr_dot = mx * rx + mz * rz   # m_hat . r (un-normalized)
    prefac = (mu0 / (4 * np.pi)) * m_moment / r**5

    dBx = prefac * (3 * mr_dot * rx - mx * r2)  # T
    dBz = prefac * (3 * mr_dot * rz - mz * r2)  # T

    return dBx * 1e9, dBz * 1e9   # convert T -> nT

# Profile setup
x_km = np.linspace(-20, 20, 401)   # km
x_m  = x_km * 1e3                  # m
depth_m  = 3000.0                  # depth to dipole, m
m_moment = 1e13                    # A·m² (representative for small magnetized body)

print('Anomaly geometry set up.')

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Compute the total-field anomaly from the vector anomaly.

def total_field_anomaly(dBx, dBz_up, I_deg):
    """
    Total-field anomaly (scalar projection approximation).

    Parameters
    ----------
    dBx : ndarray
        North component of vector anomaly, nT.
    dBz_up : ndarray
        Upward vertical component of vector anomaly, nT.
    I_deg : float
        Inclination of the regional field, degrees.

    Returns
    -------
    dF : ndarray
        Total-field anomaly, nT.  delta_F = T_hat . delta_B

    Notes
    -----
    In 2D with D=0, the regional field unit vector in (x=north, z=up) is:
        T_hat = (cos(I), -sin(I))
    because the Z-down component of the regional field corresponds to
    -z_up.  So:
        delta_F = dBx * cos(I) + dBz_up * (-sin(I))
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# At I=90° (vertical field): delta_F = -dBz_up (field is straight down, T_hat = (0,-1))
_dBx_test = np.array([0.0, 10.0, -5.0])
_dBz_test = np.array([20.0, 30.0, -8.0])
_check('delta_F at I=90° = -dBz_up', total_field_anomaly(_dBx_test, _dBz_test, 90.0),
       -_dBz_test, rtol=1e-6)

# At I=0° (horizontal field pointing north): delta_F = dBx
_check('delta_F at I=0° = dBx', total_field_anomaly(_dBx_test, _dBz_test, 0.0),
       _dBx_test, rtol=1e-6)

# Linear superposition: delta_F is linear in (dBx, dBz)
_dF1 = total_field_anomaly(np.array([1.0]), np.array([0.0]), 45.0)
_dF2 = total_field_anomaly(np.array([0.0]), np.array([1.0]), 45.0)
_dF3 = total_field_anomaly(np.array([1.0]), np.array([1.0]), 45.0)
_check('Linearity', _dF3[0], _dF1[0] + _dF2[0], rtol=1e-10)

### Task 3.1 — Anomaly shape vs. inclination

Compute and plot the total-field anomaly $\Delta F$ for the buried dipole
at **five inclinations**: $I = 90°, 60°, 45°, 30°, 0°$.
Plot all five curves on the same axes.

Requirements:
- x-axis: profile distance in km
- y-axis: $\Delta F$ in nT
- Label each curve with its inclination
- Note the location of the body ($x = 0$) with a vertical dashed line


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────

# YOUR CODE HERE

### Question 3.1 — Effect of inclination on anomaly shape

Describe how the anomaly shape changes as inclination decreases from 90° to 0°.
At $I = 90°$ (magnetic pole), what does the anomaly look like?
At $I = 0°$ (magnetic equator), what does it look like?
Why does the asymmetry increase as $I$ decreases?

**Your response:**

> *(Write 5–7 sentences. Be specific about the sign and location of anomaly features.)*


### Question 3.2 — Implications for interpretation

Given only the $\Delta F$ profile at an unknown inclination, could you uniquely
determine the location of the buried body from the peak of $\Delta F$?  
What operation would you apply to the data to make interpretation simpler?
(Hint: this operation is covered in Lab 6.)

**Your response:**

> *(Write 3–5 sentences.)*


---
## Synthesis

### S1 — Gravity analogy

In Lab 2 you computed the gravity anomaly of a buried sphere.  Compare that
result to the total-field magnetic anomaly of the same body (assume it is
induced-magnetized at $I = 90°$).  How are the anomaly shapes similar?  How do
they differ, and what causes the difference?

**Your response:**

> *(Write 4–6 sentences. Reference Poisson's relation from Homework 3.)*

### S2 — Limitation of the total-field approximation

The total-field anomaly formula $\Delta F \approx \hat{\mathbf{T}} \cdot \Delta\mathbf{B}$
is a linearization valid when $|\Delta\mathbf{B}| \ll |\mathbf{F}_0|$.  
Estimate the percentage error of this approximation if $|\Delta\mathbf{B}| = 1000$ nT
and $F_0 = 50{,}000$ nT.  Under what real-world conditions could the approximation
break down, and what would you observe in the data?

**Your response:**

> *(Write 4–5 sentences.)*

### S3 — GAD and paleomagnetism

The GAD hypothesis (that the time-averaged geomagnetic field equals a geocentric
axial dipole) is foundational to paleomagnetism.  If you measured the inclination
of remnant magnetism in a rock sample and found $I = 45°$, what paleolatitude
would you infer?  What assumptions are you making, and what could invalidate them?

**Your response:**

> *(Write 4–6 sentences. Use the inclination formula you derived.)*


---
## Extensions *(optional)*

### E1 — Tilted dipole

The geomagnetic dipole is tilted ~11° from the rotation axis, with the
geomagnetic north pole currently near 80°N, 108°W.  Extend your `gad_field`
function to accept an arbitrary pole latitude and longitude, and compute the
field on a global grid.  How much does the tilt affect $F$, $I$, and $D$
compared to the perfectly axial dipole?

### E2 — Remanent vs. induced magnetization

Redo Part 3 with the body magnetized in a **fixed direction** (remanent
magnetization, e.g., pointing north) regardless of the regional field inclination.
How does the anomaly shape differ from the induced case?  Why does this matter
for interpreting ancient crustal anomalies?

### E3 — Köenigsberger ratio

In practice, a rock's total magnetization is the vector sum of induced and
remanent components.  The Köenigsberger ratio $Q = J_r / J_i$ measures their
relative magnitudes.  For $Q = 0.5$ and $Q = 5$, compute and plot the total-field
anomaly for a buried body with a remanent direction 90° away from the regional
field.  Show how the anomaly peak shifts.


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Extension workspace.

# YOUR CODE HERE